## Step 2: Data Wrangling

This notebook takes the file we built in Step 1 (`Nhanes_Mapped.xlsx`) and turns it into one clean table we can use to train models.

#### Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np

#### Data Path

In [ ]:
BASE_DIR = "."
RAW_DIR      = os.path.join(BASE_DIR, "Nhanes_RawData")
MAPPED_XLSX  = os.path.join(RAW_DIR, "Nhanes_Mapped.xlsx")     
CLEAN_CSV    = os.path.join(RAW_DIR, "Nhanes_Cleaned.csv")   

#### Load the data

In [3]:
codebook = pd.read_excel(MAPPED_XLSX, sheet_name="Codebook")
raw = pd.read_excel(MAPPED_XLSX, sheet_name="Merged_Decoded")

print("codebook rows:", codebook.shape)
print("raw data shape:", raw.shape)
raw.head()

codebook rows: (201, 6)
raw data shape: (11933, 71)


,SEQN,ALQ111,ALQ121,ALQ130,ALQ142,ALQ270,ALQ280,ALQ151,ALQ170,SDDSRVYR,...,SLQ330,SLD013,SMQ020,SMQ040,SMD641,SMD650,SMD100MN,SMQ621,SMD630,SMAQUEX2
0,130378,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,...,09:00,9.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,1.0
1,130379,1.0,2.0,3.0,0.0,NaN,NaN,2.0,NaN,12,...,06:00,9.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,1.0
2,130380,1.0,10.0,1.0,0.0,NaN,NaN,2.0,NaN,12,...,09:00,9.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3,130381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Step 2.1: Clean up 
NHANES uses numbers like `7`, `9`, `77`, `99` to mean "Refused" or "Don't know" instead of a real answer. If we leave these in, a model might think someone who "refused" scored super high on a question. So the first real cleaning step is: for every column, look up (in the codebook) which codes mean Refused/Don't know, and turn those into `NaN` (missing) instead.

In [4]:
# build a dictionary: {column_name: set of codes that mean "Refused" or "Don't know"}
missing_code_map = {}
for _, row in codebook.iterrows():
    if row["Meaning"] in ("Refused", "Don't know"):
        missing_code_map.setdefault(row["Variable"], set()).add(row["Code"])

print(f"Found Refused/Don't know codes for {len(missing_code_map)} columns.")

df = raw.copy()
for col, codes in missing_code_map.items():
    if col in df.columns:
        df[col] = df[col].replace(list(codes), np.nan)

print("Done. Data shape is unchanged (still just replacing values):", df.shape)

Found Refused/Don't know codes for 22 columns.
Done. Data shape is unchanged (still just replacing values): (11933, 71)


#### Step 2.2: Build our target: a depression flag from the PHQ-9
The PHQ-9 is 9 questions (`DPQ010`–`DPQ090`), each scored 0-3 (higher = worse). Adding all 9 up gives a score from 0 to 27. This is a well-known, standard screening tool, and a score of **10 or more is the standard cutoff** used in research to flag likely depression.

Rule we use: if someone is missing *any* of the 9 answers, we can't trust their total score, so their score (and flag) is left as missing rather than guessed.

In [ ]:
phq_cols = [f"DPQ0{n}0" for n in range(1, 10)]  
print("PHQ-9 columns:", phq_cols)

answered_all_9 = df[phq_cols].notna().all(axis=1)
df["PHQ9_Score"] = df[phq_cols].sum(axis=1, min_count=9)   # min_count=9 -> NaN unless all 9 present
df.loc[~answered_all_9, "PHQ9_Score"] = np.nan

# Standard clinical cutoff: score >= 10 means likely depression
df["Depression_Flag"] = np.where(
    df["PHQ9_Score"].isna(), np.nan,
    np.where(df["PHQ9_Score"] >= 10, 1, 0)
)

print(df["PHQ9_Score"].describe())
print()
print("Depression_Flag counts (1 = likely depression, 0 = not, NaN = we can't tell):")
print(df["Depression_Flag"].value_counts(dropna=False))

PHQ-9 columns: ['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']
count    5455.000000
mean        4.133272
std         4.765574
min         0.000000
25%         1.000000
50%         2.000000
75%         6.000000
max        26.000000
Name: PHQ9_Score, dtype: float64

Depression_Flag counts (1 = likely depression, 0 = not, NaN = we can't tell):
Depression_Flag
NaN    6478
0.0    4732
1.0     723
Name: count, dtype: int64


#### Step 2.3: Keep only rows where we know the answer
We can only train a model on people where we actually know if they show depression indicators or not, so we drop anyone with a missing `Depression_Flag`.

In [6]:
clean = df[df["Depression_Flag"].notna()].copy()
print("Rows before:", df.shape[0], " -> Rows after keeping only a known target:", clean.shape[0])

print()
print("Class balance (as %):")
print((clean["Depression_Flag"].value_counts(normalize=True) * 100).round(1))

Rows before: 11933  -> Rows after keeping only a known target: 5455

Class balance (as %):
Depression_Flag
0.0    86.7
1.0    13.3
Name: proportion, dtype: float64


**Note:** only about 1 in 8 people are flagged as likely depression. This is an *imbalanced* target, keep this in mind for the modeling notebook (e.g. use `stratify=` when splitting train/test, and don't rely on plain accuracy alone, precision/recall/F1 matter more here).

#### Step 2.4: Check how much data is missing in each column
Some columns (mostly ones with built-in skip logic, like smoking-detail questions asked only to smokers) are missing for most people. A column that's missing for, say, 95% of people isn't very useful as a predictor, so we'll drop columns above a missingness threshold.

In [7]:
missing_pct = clean.isna().mean().sort_values(ascending=False) * 100
missing_pct.round(1).to_frame("percent_missing")

,percent_missing
RIDAGEMN,100.0
SMQ621,100.0
SMD630,100.0
DMDHSEDZ,98.2
SMD641,97.2
...,...
DPQ090,0.0
DPQ070,0.0
SMAQUEX2,0.0
PHQ9_Score,0.0


#### Step 2.5: Decide which columns to drop
We drop 4 kinds of columns:
- **Too much missing data** (more than 50% missing) — not reliable enough to use.
- **The PHQ-9 questions themselves** (`DPQ010`-`DPQ090`, `DPQ100`) — we used these to *build* our target, so using them again as predictors would be "cheating" (this is called data leakage — the model would just learn to re-read the target instead of finding real patterns).
- **Survey design/weighting columns** (`WTINT2YR`, `WTMEC2YR`, `SDMVSTRA`, `SDMVPSU`, `SDDSRVYR`, `RIDSTATR`) — these describe how NHANES samples people, not a person's health/lifestyle, so they aren't useful lifestyle predictors for this project.
- **Redundant sleep clock-times** (`SLQ300`, `SLQ310`, `SLQ320`, `SLQ330`) — these are text times like "21:30", and we already have the same information as plain numbers of sleep hours in `SLD012`/`SLD013`.
- **A frequency number whose unit was already dropped** (`PAD810Q`) — this is "how often do you do vigorous exercise", but the answer only makes sense together with its unit column, `PAD810U` (day/week/month/year). We're already dropping `PAD810U` for being >50% missing, so a leftover number like "3" in `PAD810Q` could mean 3 times a day, a week, a month, or a year — we can't tell, so it's not a usable number on its own and should go too.

In [8]:
high_missing_cols = missing_pct[missing_pct > 50].index.tolist()
leakage_cols = phq_cols + ["DPQ100"]
design_cols = ["WTINT2YR", "WTMEC2YR", "SDMVSTRA", "SDMVPSU", "SDDSRVYR", "RIDSTATR"]
redundant_cols = ["SLQ300", "SLQ310", "SLQ320", "SLQ330"]
orphaned_cols = ["PAD810Q"]   # its unit column (PAD810U) is already being dropped above, see note

drop_cols = set(high_missing_cols) | set(leakage_cols) | set(design_cols) | set(redundant_cols) | set(orphaned_cols)
drop_cols = [c for c in drop_cols if c in clean.columns]

print(f"Dropping {len(drop_cols)} columns:")
print(sorted(drop_cols))

model_df = clean.drop(columns=drop_cols)
print()
print("Shape after dropping columns:", model_df.shape)

Dropping 42 columns:
['ALQ170', 'ALQ270', 'ALQ280', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRGND', 'DMDHRMAZ', 'DMDHSEDZ', 'DMDYRUSR', 'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090', 'DPQ100', 'IND310', 'PAD810Q', 'PAD810U', 'PAD820', 'RIDAGEMN', 'RIDEXAGM', 'RIDEXPRG', 'RIDSTATR', 'SDDSRVYR', 'SDMVPSU', 'SDMVSTRA', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'SMD100MN', 'SMD630', 'SMD641', 'SMD650', 'SMQ040', 'SMQ621', 'WTINT2YR', 'WTMEC2YR']

Shape after dropping columns: (5455, 31)


#### Step 2.6: Fill in the small amount of missing data that's left
Using the codebook, we already know which columns are **categorical** (a code stands for a category, like Male/Female) and which are **continuous** (a plain number, like age or hours of sleep). We fill missing values differently for each type:
- **Continuous** columns → fill with the **median** (middle value), a common, simple choice that isn't thrown off by outliers.
- **Categorical** columns → fill with `-1`, a brand new code meaning "not answered", so we don't accidentally pretend someone picked a real category they didn't.

In [9]:
continuous_vars = set(codebook.loc[codebook["Meaning"] == "continuous / numeric", "Variable"])
id_vars = set(codebook.loc[codebook["Meaning"] == "respondent ID", "Variable"])
categorical_vars = set(codebook["Variable"]) - continuous_vars - id_vars

remaining_continuous = [c for c in model_df.columns if c in continuous_vars]
remaining_categorical = [c for c in model_df.columns if c in categorical_vars]

print("Continuous columns to fill with median:", remaining_continuous)
print()
print("Categorical columns to fill with -1 ('not answered'):", remaining_categorical)

for col in remaining_continuous:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in remaining_categorical:
    model_df[col] = model_df[col].fillna(-1)

still_missing = model_df.isna().sum().sum()
print()
print(f"Missing values left in the whole table: {still_missing} (should be 0)")

Continuous columns to fill with median: ['ALQ130', 'ALQ142', 'RIDAGEYR', 'DMDHHSIZ', 'INDFMPIR', 'INDFMMPI', 'PAD790Q', 'PAD800', 'PAD680', 'SLD012', 'SLD013']

Categorical columns to fill with -1 ('not answered'): ['ALQ111', 'ALQ121', 'ALQ151', 'RIAGENDR', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMQMILIZ', 'DMDBORN4', 'DMDEDUC2', 'DMDMARTZ', 'HSQ590', 'INDFMMPC', 'INQ300', 'PAD790U', 'SMQ020', 'SMAQUEX2']

Missing values left in the whole table: 0 (should be 0)


#### Step 2.7: Quick sanity check on the final table
Let's take a final look before saving: shape, column list, and the first few rows.

In [10]:
print("Final shape:", model_df.shape)
print()
print("Final columns:")
print(model_df.columns.tolist())
model_df.head()

Final shape: (5455, 31)

Final columns:
['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ151', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMQMILIZ', 'DMDBORN4', 'DMDEDUC2', 'DMDMARTZ', 'DMDHHSIZ', 'INDFMPIR', 'HSQ590', 'INDFMMPI', 'INDFMMPC', 'INQ300', 'PAD790Q', 'PAD790U', 'PAD800', 'PAD680', 'SLD012', 'SLD013', 'SMQ020', 'SMAQUEX2', 'PHQ9_Score', 'Depression_Flag']


,SEQN,ALQ111,ALQ121,ALQ130,ALQ142,ALQ151,RIAGENDR,RIDAGEYR,RIDRETH1,RIDRETH3,...,PAD790Q,PAD790U,PAD800,PAD680,SLD012,SLD013,SMQ020,SMAQUEX2,PHQ9_Score,Depression_Flag
1,130379,1.0,2.0,3.0,0.0,2.0,1,66,3,3,...,4.0,W,45.0,480.0,9.0,9.0,1.0,1.0,1.0,0.0
2,130380,1.0,10.0,1.0,0.0,2.0,2,44,2,2,...,1.0,W,20.0,240.0,8.0,9.0,2.0,1.0,2.0,0.0
8,130386,1.0,4.0,2.0,10.0,2.0,1,34,1,1,...,1.0,W,30.0,180.0,7.5,8.0,1.0,1.0,1.0,0.0
9,130387,1.0,0.0,2.0,4.0,2.0,2,68,3,3,...,0.0,-1,60.0,1200.0,3.0,5.0,2.0,1.0,0.0,0.0
11,130389,1.0,2.0,2.0,10.0,2.0,1,59,3,3,...,3.0,W,45.0,720.0,8.0,8.0,1.0,1.0,0.0,0.0


#### Step 2.8: Save the cleaned, model-ready table
This is the file the next notebook (model building) will load. `SEQN` is kept as a respondent ID (useful for joining more data later, or debugging) — it should be dropped or ignored as a *feature* before training, since it's just an ID number, not something that describes the person.

In [11]:
model_df.to_csv(CLEAN_CSV, index=False)
print(f"Saved {CLEAN_CSV}")
print(f"Final cleaned dataset: {model_df.shape[0]} rows, {model_df.shape[1]} columns")

Saved .\Nhanes_RawData\Nhanes_Cleaned.csv
Final cleaned dataset: 5455 rows, 31 columns
